# GOT-OCR 2.0 — Colab API Server

Bu notebook GOT-OCR 2.0 modelini Colab'da çalıştırarak bir API sunucusu oluşturur.

**Adımlar:**
1. Runtime → Change runtime type → **T4 GPU** seç
2. Tüm hücreleri sırasıyla çalıştır
3. Son hücrede `ngrok` URL'ini kopyala
4. Bu URL'yi Streamlit app'indeki 'GOT-OCR API URL' alanına yapıştır

**ÖNEMLİ:** GPU olmadan çalışmaz!

In [ ]:
# 1. Bağımlılıkları kur (transformers 4.37.2 ZORUNLU)
!pip install -q transformers==4.37.2 verovio fastapi uvicorn python-multipart pyngrok Pillow

# torch'u YUKARI KURMA — Colab'ın kendi CUDA versiyonunu kullan
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU bulunamadı! Runtime → Change runtime type → T4 GPU seçin.')

In [ ]:
# 2. GOT-OCR 2.0 modelini yükle
from transformers import AutoModelForCausalLM, AutoProcessor
import torch

MODEL_ID = 'stepfun-ai/GOT-OCR2_0'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

print(f'Model yükleniyor... (device={DEVICE}, dtype={DTYPE})')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True
).to(DEVICE)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print('Model yüklendi!')

In [ ]:
# 3. OCR fonksiyonu
from PIL import Image
import io, base64

def run_ocr(image_bytes, max_new_tokens=4096):
    """GOT-OCR ile Z raporu oku"""
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    
    # Z raporu içinPlainText output kullan
    prompt = 'OCR with format: plain text'
    
    inputs = processor(image, prompt, return_tensors='pt').to(DEVICE, dtype=DTYPE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    
    # Sadece generated portion
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    result = processor.decode(generated, skip_special_tokens=True)
    return result

print('OCR fonksiyonu hazır!')

In [ ]:
# 4. FastAPI sunucusu oluştur
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import nest_asyncio

nest_asyncio.apply()

app = FastAPI(title='GOT-OCR API')

# CORS — Streamlit Cloud izin ver
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/health')
async def health():
    return {'status': 'ok', 'model': 'GOT-OCR2_0', 'device': DEVICE}

@app.post('/ocr')
async def ocr_endpoint(file: UploadFile = File(...)):
    try:
        contents = await file.read()
        if len(contents) > 10 * 1024 * 1024:  # 10MB limit
            raise HTTPException(status_code=400, detail='Dosya çok büyük (max 10MB)')
        result = run_ocr(contents)
        return {'text': result, 'status': 'ok'}
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print('FastAPI sunucusu hazır!')

In [ ]:
# 5. ngrok ile herkese açık URL oluştur ve sunucuyu başlat
from pyngrok import ngrok
import threading, time

# ngrok auth token — https://dashboard.ngrok.com/get-started/your-authtoken
# Ücretsiz hesap ile çalışır, ama URL her restart'ta değişir
NGROK_TOKEN = ''  # Boş bırakabilirsin, ücretsiz çalışır

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Port 8000'de tünel aç
public_url = ngrok.connect(8000, 'http')
print(f'\n{"="*60}')
print(f'GOT-OCR API ACTVe!')
print(f'Public URL: {public_url}')
print(f'\nBu URLyi Streamlit appindeki "GOT-OCR API URL" alinine yapistirin.')
print(f'{"="*60}\n')

# Sunucuyu arka planda başlat
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print('Sunucu baslatildi! Test icin asagidaki hucreyi calistirin.')

In [ ]:
# 6. Test — bu hücreyi çalıştırarak API'nin çalıştığını doğrula
import requests

# Health check
try:
    r = requests.get(f'{public_url}/health', timeout=5)
    print(f'Health: {r.json()}')
except Exception as e:
    print(f'Hata: {e}')
    print('Sunucu henüz hazır değil, 5sn bekleyip tekrar deneyin.')

In [ ]:
# 7. durdurmak istersen
# ngrok.kill()
print('Sunucu calisiyor. Durdurmak icin Colab runtime i durdurun.')